# T55-企业预警通导入重点指标

## 项目简介
本项目用于将Excel中的重点指标数据导入到数据库中，支持trade_code替换和数据清洗。

---
## 第一章：环境配置与依赖导入

In [ ]:
# 第一章：环境配置与依赖导入
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text
import os
import warnings
warnings.filterwarnings('ignore')

# 导入配置
from config import DB_BOND_URL, PG_URL, DATA_DIR, OUTPUT_DIR
from src import ExcelReader, DataImporter, TradeCodeMapper

print("依赖导入完成！")

---
## 第二章：数据库连接与数据准备

In [ ]:
# 第二章：数据库连接与数据准备
# 创建数据库连接
sql_engine_bond = sqlalchemy.create_engine(DB_BOND_URL, poolclass=sqlalchemy.pool.NullPool)
cursor_bond = sql_engine_bond.connect()

print("数据库连接成功！")

# 初始化模块
excel_reader = ExcelReader(DATA_DIR)
data_importer = DataImporter(sql_engine_bond)
trade_code_mapper = TradeCodeMapper(sql_engine_bond)

print("模块初始化完成！")

---
## 第三章：Excel数据读取

In [ ]:
# 第三章：Excel数据读取
# 列出所有Excel文件
excel_files = excel_reader.list_excel_files()
print(f"找到 {len(excel_files)} 个Excel文件:")
for f in excel_files:
    print(f"  - {f}")

# 读取第一个Excel文件（示例）
if excel_files:
    df_raw = excel_reader.read_excel(excel_files[0])
    print(f"\n读取文件: {excel_files[0]}")
    print(f"数据形状: {df_raw.shape}")
    print(f"\n列名:")
    print(df_raw.columns.tolist())
    print(f"\n前5行数据:")
    display(df_raw.head())
else:
    print("未找到Excel文件，请将文件放入data目录")
    df_raw = pd.DataFrame()

---
## 第四章：数据清洗与预处理

In [ ]:
# 第四章：数据清洗与预处理
if not df_raw.empty:
    # 清理列名
    df_clean = excel_reader.clean_column_names(df_raw)
    print("列名清理完成")
    
    # 查看数据信息
    info = excel_reader.get_column_info(df_clean)
    print(f"\n数据信息:")
    print(f"  - 行数: {info['shape'][0]}")
    print(f"  - 列数: {info['shape'][1]}")
    print(f"  - 空值统计:")
    for col, count in info['null_counts'].items():
        if count > 0:
            print(f"    {col}: {count}")
    
    # 显示数据类型
    print(f"\n数据类型:")
    print(df_clean.dtypes)
else:
    df_clean = pd.DataFrame()
    print("无数据需要清洗")

---
## 第五章：Trade Code映射替换

In [ ]:
# 第五章：Trade Code映射替换
if not df_clean.empty:
    # 检查是否有发行人名称列
    name_column = None
    possible_names = ['ths_issuer_name_cn_bond', '发行人名称', '公司名称', '企业名称']
    for col in possible_names:
        if col in df_clean.columns:
            name_column = col
            break
    
    if name_column:
        print(f"找到发行人名称列: {name_column}")
        
        # 执行映射
        df_mapped = trade_code_mapper.map_trade_codes(df_clean, name_column, 'trade_code')
        
        # 统计映射结果
        total = len(df_mapped)
        mapped = df_mapped['trade_code'].notna().sum()
        print(f"\n映射统计:")
        print(f"  - 总行数: {total}")
        print(f"  - 成功映射: {mapped}")
        print(f"  - 未映射: {total - mapped}")
        print(f"  - 映射率: {mapped/total*100:.2f}%")
        
        # 过滤包含中文的trade_code
        df_filtered = trade_code_mapper.filter_chinese_trade_codes(df_mapped, 'trade_code')
        print(f"\n过滤中文trade_code后: {len(df_filtered)} 行")
        
        # 过滤空值
        if 'value' in df_filtered.columns:
            df_filtered = trade_code_mapper.filter_empty_values(df_filtered, ['trade_code', 'value'])
            print(f"过滤空值后: {len(df_filtered)} 行")
    else:
        print("未找到发行人名称列，跳过trade_code映射")
        df_filtered = df_clean
else:
    df_filtered = pd.DataFrame()
    print("无数据需要映射")

---
## 第六章：数据导入到数据库

In [ ]:
# 第六章：数据导入到数据库
if not df_filtered.empty:
    # 目标表名
    table_name = '指标数据1'
    
    # 检查表是否存在
    try:
        table_info = data_importer.get_table_info(table_name)
        print(f"目标表信息:")
        print(f"  - 表名: {table_info['table_name']}")
        print(f"  - 现有行数: {table_info['row_count']}")
        print(f"  - 列: {table_info['columns']}")
    except Exception as e:
        print(f"表不存在或查询失败: {e}")
    
    # 导入数据
    try:
        rows_imported = data_importer.import_to_table(df_filtered, table_name, if_exists='append')
        print(f"\n成功导入 {rows_imported} 行数据到表 {table_name}")
    except Exception as e:
        print(f"导入失败: {e}")
else:
    print("无数据需要导入")

---
## 第七章：SQL后处理与验证

In [ ]:
# 第七章：SQL后处理与验证
# 执行trade_code替换SQL
sql_replace = """
WITH trade_code_mapping AS (
    SELECT MAX(trade_code) AS trade_code, ths_issuer_name_cn_bond
    FROM (
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_credit
        UNION ALL
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_finance
        UNION ALL
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_abs
    ) SQ
    GROUP BY ths_issuer_name_cn_bond
)
UPDATE 指标数据1 A
SET trade_code = trade_code_mapping.trade_code
FROM trade_code_mapping
WHERE A.ths_issuer_name_cn_bond = trade_code_mapping.ths_issuer_name_cn_bond;
"""

# 删除中文trade_code的SQL
sql_delete_chinese = "DELETE FROM 指标数据1 WHERE trade_code ~ '[\u4e00-\u9fa5]'"

# 删除空值的SQL
sql_delete_empty = "DELETE FROM 指标数据1 WHERE value IS NULL OR value = ''"

# 执行后处理
try:
    # 注意：以下SQL需要根据实际数据库类型调整语法
    print("执行后处理SQL...")
    
    # 验证数据
    verify_sql = "SELECT COUNT(*) as cnt FROM 指标数据1"
    result = pd.read_sql(verify_sql, cursor_bond)
    print(f"\n表中现有数据: {result['cnt'].iloc[0]} 行")
    
    # 显示样本数据
    sample_sql = "SELECT * FROM 指标数据1 LIMIT 10"
    sample = pd.read_sql(sample_sql, cursor_bond)
    print("\n样本数据:")
    display(sample)
    
except Exception as e:
    print(f"后处理失败: {e}")

# 关闭连接
cursor_bond.close()
print("\n数据库连接已关闭")

# 保存处理日志
log_file = os.path.join(OUTPUT_DIR, 'import_log.txt')
with open(log_file, 'w', encoding='utf-8') as f:
    f.write(f"导入时间: {pd.Timestamp.now()}\n")
    f.write(f"导入行数: {len(df_filtered) if not df_filtered.empty else 0}\n")
print(f"\n日志已保存到: {log_file}")